# LangChain: Memory

## Outline
* ConversationBufferMemory： 存储完整的对话历史（最简单）。
* ConversationBufferWindowMemory： 存储最近 N 轮的对话历史（控制长度）。
* ConversationTokenBufferMemory： 根据 Token 数量限制历史记录（更精确）。
* ConversationSummaryMemory： 总结旧的对话历史，只存储摘要（节省 Token）。

## ConversationBufferMemory
存储完整的对话历史（最简单）

In [1]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

import warnings
warnings.filterwarnings('ignore')

### 说明
过时方法已经迁移到langchain-classic中，导入改包，改引入路径，老的方法任然能使用。


In [2]:
from langchain_openai import ChatOpenAI
# ConversationChain已经过时，但是能运行
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

In [ ]:
# 实例化 LLM 模型，使用 ChatOpenAI 类，但将其重定向到 DashScope 接口
llm = ChatOpenAI(
    temperature=0.0, 
    model="qwen-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.getenv("DASHSCOPE_API_KEY")
    )

# 实例化记忆模块：ConversationBufferMemory 存储完整的对话历史
memory = ConversationBufferMemory()

# 实例化 ConversationChain（旧版LangChain的封装类，用于管理对话流程）
# 它内部自动化了记忆的加载和保存过程。
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

C:\Users\bangsun\AppData\Local\Temp\ipykernel_35948\1506788229.py:8: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
C:\Users\bangsun\AppData\Local\Temp\ipykernel_35948\1506788229.py:10: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(


In [ ]:
# --- 运行对话流程，并测试记忆 ---

# 第一次对话：Chain 内部自动加载记忆（此时为空），调用 LLM，然后保存新的对话记录。
conversation.predict(input="Hi, my name is Andrew")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi, my name is Andrew
AI:

> Finished chain.


"Hello Andrew! It's a pleasure to meet you. I'm Qwen, an AI assistant created by Alibaba Cloud. How can I assist you today? Is there anything specific you're interested in or any questions you have?"

In [ ]:
# 第二次对话：Chain 内部自动加载第一次对话的记录，调用 LLM，然后保存新的记录。
conversation.predict(input="What is 1+1?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! It's a pleasure to meet you. I'm Qwen, an AI assistant created by Alibaba Cloud. How can I assist you today? Is there anything specific you're interested in or any questions you have?
Human: What is 1+1?
AI:

> Finished chain.


"1+1 equals 2! It's one of the most basic and fundamental equations in arithmetic. If you have any more math questions or need help with anything else, feel free to ask!"

In [ ]:
# 第三次对话：Chain 内部再次加载完整的历史记录，LLM 会知道用户叫 Andrew。
conversation.predict(input="What is my name?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi, my name is Andrew
AI: Hello Andrew! It's a pleasure to meet you. I'm Qwen, an AI assistant created by Alibaba Cloud. How can I assist you today? Is there anything specific you're interested in or any questions you have?
Human: What is 1+1?
AI: 1+1 equals 2! It's one of the most basic and fundamental equations in arithmetic. If you have any more math questions or need help with anything else, feel free to ask!
Human: What is my name?
AI:

> Finished chain.


"Your name is Andrew! If you have any more questions or if there's anything else I can help you with, feel free to let me know."

In [ ]:
# 打印 memory.buffer：查看 memory 对象内部存储的原始对话字符串
print(memory.buffer)

Human: Hi, my name is Andrew
AI: Hello Andrew! It's a pleasure to meet you. I'm Qwen, an AI assistant created by Alibaba Cloud. How can I assist you today? Is there anything specific you're interested in or any questions you have?
Human: What is 1+1?
AI: 1+1 equals 2! It's one of the most basic and fundamental equations in arithmetic. If you have any more math questions or need help with anything else, feel free to ask!
Human: What is my name?
AI: Your name is Andrew! If you have any more questions or if there's anything else I can help you with, feel free to let me know.


In [ ]:
# 再次手动加载记忆变量（在实际 Chain 中，这个操作是自动发生的）
# 从记忆存储中加载当前完整的对话历史（或状态），并将其格式化为字典，以便插入到 LLM 的 Prompt 中。
memory.load_memory_variables({})

{'history': "Human: Hi, my name is Andrew\nAI: Hello Andrew! It's a pleasure to meet you. I'm Qwen, an AI assistant created by Alibaba Cloud. How can I assist you today? Is there anything specific you're interested in or any questions you have?\nHuman: What is 1+1?\nAI: 1+1 equals 2! It's one of the most basic and fundamental equations in arithmetic. If you have any more math questions or need help with anything else, feel free to ask!\nHuman: What is my name?\nAI: Your name is Andrew! If you have any more questions or if there's anything else I can help you with, feel free to let me know."}

In [ ]:
# 重新实例化（Re-instantiate）一个全新的 ConversationBufferMemory 对象，并将其赋值给 memory 变量。
# 会清除之前的所有记忆
memory = ConversationBufferMemory()

In [ ]:
# 手动保存上下文：记录第二组对话，它会被追加到上一次的记录后面
memory.save_context({"input": "Hi"}, 
                    {"output": "What's up"})

In [ ]:
# 打印 memory.buffer：查看新的手动记录
print(memory.buffer)

Human: Hi
AI: What's up


In [ ]:
# 再次手动加载记忆变量（在实际 Chain 中，这个操作是自动发生的）
memory.load_memory_variables({})

{'history': "Human: Hi\nAI: What's up"}

In [ ]:
# 手动保存上下文：记录第二组对话，它会被追加到上一次的记录后面
memory.save_context({"input": "Not much, just hanging"}, 
                    {"output": "Cool"})

In [18]:
memory.load_memory_variables({})

{'history': "Human: Hi\nAI: What's up\nHuman: Not much, just hanging\nAI: Cool"}

## ConversationBufferWindowMemory
存储最近 N 轮的对话历史（控制长度）。

In [20]:
from langchain_classic.memory import ConversationBufferWindowMemory

In [21]:
memory = ConversationBufferWindowMemory(k=1)               

C:\Users\bangsun\AppData\Local\Temp\ipykernel_35948\802562642.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferWindowMemory(k=1)


In [22]:
memory.save_context({"input": "Hi"},
                    {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})


In [23]:
memory.load_memory_variables({})

{'history': 'Human: Not much, just hanging\nAI: Cool'}

In [24]:
# llm = ChatOpenAI(temperature=0.0, model=llm_model)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=False
)

In [25]:
conversation.predict(input="Hi, my name is Andrew")

"Hello Andrew! It's great to meet you. I'm Qwen, an AI assistant created by Alibaba Cloud. How can I assist you today? Is there anything specific on your mind or any topic you'd like to discuss?"

In [26]:
conversation.predict(input="What is 1+1?")

"1+1 equals 2! It's one of the most basic and fundamental equations in arithmetic. If you have any more math questions or need help with anything else, feel free to ask!"

In [27]:
conversation.predict(input="What is my name?")

"I'm sorry, but I don't have any information about your name. Since we're just starting our conversation, I haven't been given any details about you. If you'd like to share your name, I'd be happy to use it in our conversation!"

## ConversationTokenBufferMemory
根据 Token 数量限制历史记录（更精确）

In [ ]:
#!pip install tiktoken

In [44]:
from langchain_classic.memory import ConversationTokenBufferMemory
from langchain_community.chat_models.tongyi import ChatTongyi
# 初始化模型,由于会下载分词配置会失败，所以需要用专门的ChatTongyi，不然会调用openai的会报错
tongyiLLM = ChatTongyi(
    model="qwen-turbo",  # 或其他通义千问模型
    dashscope_api_key=os.getenv("DASHSCOPE_API_KEY")  # 通义千问 API key
)

In [ ]:
memory = ConversationTokenBufferMemory(llm=tongyiLLM, max_token_limit=50)
memory.save_context({"input": "AI is what?!"},
                    {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"},
                    {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, 
                    {"output": "Charming!"})
memory.save_context({"input": "what is your name ?"}, 
                    {"output": "Charming! how are you? I am fine , thank you and you"})

In [35]:
memory.load_memory_variables({})

{'history': 'Human: Backpropagation is what?\nAI: Beautiful!\nHuman: Chatbots are what?\nAI: Charming!\nHuman: what is your name ?\nAI: Charming! how are you? I am fine , thank you and you'}

## ConversationSummaryMemory

总结旧的对话历史，只存储摘要（节省 Token）

In [37]:
from langchain_classic.memory import ConversationSummaryBufferMemory

In [39]:
# create a long string
schedule = "There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian resturant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."

memory = ConversationSummaryBufferMemory(llm=tongyiLLM, max_token_limit=100)
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"},
                    {"output": "Cool"})
memory.save_context({"input": "What is on the schedule today?"}, 
                    {"output": f"{schedule}"})

In [40]:
memory.load_memory_variables({})

{'history': "System: The human says hello. The AI asks what's up. The human says not much, just hanging. The AI says cool. The human asks what is on the schedule today.\nAI: There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo."}

In [41]:
conversation = ConversationChain(
    llm=llm, 
    memory = memory,
    verbose=True
)

In [42]:
conversation.predict(input="What would be a good demo to show?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
System: The human says hello. The AI asks what's up. The human says not much, just hanging. The AI says cool. The human asks what is on the schedule today.
AI: There is a meeting at 8am with your product team. You will need your powerpoint presentation prepared. 9am-12pm have time to work on your LangChain project which will go quickly because Langchain is such a powerful tool. At Noon, lunch at the italian resturant with a customer who is driving from over an hour away to meet you to understand the latest in AI. Be sure to bring your laptop to show the latest LLM demo.
Human: What would be a good demo to show?
AI:

> Finished chain.


"A great demo to show would be one that highlights the capabilities and potential of the latest Large Language Models (LLMs) in a way that's both impressive and relevant to your customer. Here are a few ideas:\n\n1. **Interactive Q&A Session:**\n   - **Description:** Demonstrate the LLM's ability to answer complex, domain-specific questions in real-time. This could be particularly useful if your customer is from a specific industry (e.g., healthcare, finance, or technology).\n   - **How to Prepare:** Prepare a list of challenging questions related to the customer's industry or area of interest. You can also allow the customer to ask their own questions on the spot.\n\n2. **Text Summarization:**\n   - **Description:** Show how the LLM can summarize long documents, articles, or reports into concise, key points. This is especially useful for busy professionals who need to quickly grasp the essence of lengthy content.\n   - **How to Prepare:** Bring a few sample documents or articles that 

In [43]:
memory.load_memory_variables({})

{'history': "System: The human says hello. The AI asks what's up. The human says not much, just hanging. The AI says cool. The human asks what is on the schedule today. The AI outlines the day's schedule, including a meeting at 8am with the product team, time to work on the LangChain project, and lunch at a local Italian restaurant with a customer. The human asks what would be a good demo to show, and the AI provides several detailed ideas for demonstrating the capabilities of the latest Large Language Models (LLMs), including interactive Q&A, text summarization, code generation, personalized content creation, language translation, sentiment analysis, and a customized chatbot."}

Reminder: Download your notebook to you local computer to save your work.

提醒：请将您的笔记本下载到本地计算机以保存您的工作